In [1]:
#Import Libraries
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# LOAD DATA
data_dir = "../data/"

folders = sorted([
    f for f in os.listdir(data_dir)
    if os.path.isdir(os.path.join(data_dir, f))
])

print("Folders found:", folders)

# Optional check
assert len(folders) == 16, "Expected 16 datasets!"


Folders found: ['1', '10', '11', '12', '13', '14', '15', '16', '2', '3', '4', '5', '6', '7', '8', '9']


In [3]:
models = {
    "SVM": (
        SVC(),
        {
            "C": [0.1, 1, 10],
            "kernel": ["rbf"],
            "gamma": ["scale", "auto"]
        }
    ),
    
    "KNN": (
        KNeighborsClassifier(),
        {
            "n_neighbors": [3, 5, 7, 9],
            "weights": ["uniform", "distance"]
        }
    ),
    
    "DecisionTree": (
        DecisionTreeClassifier(),
        {
            "max_depth": [None, 5, 10, 20],
            "min_samples_split": [2, 5, 10]
        }
    ),
    
    "RandomForest": (
        RandomForestClassifier(),
        {
            "n_estimators": [100, 200],
            "max_depth": [None, 10, 20],
            "min_samples_split": [2, 5]
        }
    ),
    
    "MLP": (
        MLPClassifier(max_iter=500),
        {
            "hidden_layer_sizes": [(50,), (100,), (100, 50)],
            "learning_rate_init": [0.001, 0.01],
            "alpha": [0.0001, 0.001]
        }
    )
}

In [4]:
final_results = []

# 10-FOLD CV
for folder in folders:
    print(f"\nProcessing Dataset {folder}...")

    train_path = os.path.join(data_dir, folder, "train.csv")
    print("Path:", train_path)

    df = pd.read_csv(train_path)

    X = df.iloc[:, :-1]
    y = df.iloc[:, -1]

    # 10-FOLD CV
    kf = KFold(n_splits=10, shuffle=True, random_state=42)

    results = {name: {"acc": [], "f1": [], "params": []} for name in models.keys()}

    # =============================
    # CROSS VALIDATION LOOP
    # =============================
    for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
        print(f"  Fold {fold}")

        X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        # =============================
        # MISSING VALUE IMPUTATION
        # =============================
        for col in X_train.columns:
            mean_val = X_train[col].mean()
            X_train[col] = X_train[col].fillna(mean_val)
            X_val[col] = X_val[col].fillna(mean_val)

        # =============================
        # SCALING
        # =============================
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)

        # =============================
        # TRAIN MODELS
        # =============================
        for name, (model, params) in models.items():
            grid = GridSearchCV(model, params, cv=3, scoring='f1_macro', n_jobs=-1)
            grid.fit(X_train, y_train)

            best_model = grid.best_estimator_
            y_pred = best_model.predict(X_val)

            acc = accuracy_score(y_val, y_pred)
            f1 = f1_score(y_val, y_pred, average='macro')

            results[name]["acc"].append(acc)
            results[name]["f1"].append(f1)
            # results[name]["params"].append(str(grid.best_params_))

            # print(f"    {name} | Acc: {acc:.4f} | F1: {f1:.4f}")

            best_params = grid.best_params_

            results[name]["params"].append(str(best_params))

            print(f"    {name} | Acc: {acc:.4f} | F1: {f1:.4f} | Params: {best_params}")

    # =============================
    # AGGREGATE RESULTS
    # =============================
    for name in models.keys():
        acc_mean = np.mean(results[name]["acc"])
        acc_std = np.std(results[name]["acc"])

        f1_mean = np.mean(results[name]["f1"])
        f1_std = np.std(results[name]["f1"])

        # Most frequent best params
        best_params = max(set(results[name]["params"]), key=results[name]["params"].count)

        final_results.append({
            "Dataset": f"Dataset_{folder}",
            "Model": name,
            "Accuracy": f"{acc_mean:.4f} ± {acc_std:.4f}",
            "F1 Score": f"{f1_mean:.4f} ± {f1_std:.4f}",
            "Best Params": best_params
        })




Processing Dataset 1...
Path: ../data/1/train.csv
  Fold 1
    SVM | Acc: 0.9677 | F1: 0.9511 | Params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
    KNN | Acc: 0.9771 | F1: 0.9647 | Params: {'n_neighbors': 3, 'weights': 'distance'}
    DecisionTree | Acc: 1.0000 | F1: 1.0000 | Params: {'max_depth': None, 'min_samples_split': 2}
    RandomForest | Acc: 0.9979 | F1: 0.9974 | Params: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
    MLP | Acc: 0.9896 | F1: 0.9869 | Params: {'alpha': 0.001, 'hidden_layer_sizes': (100, 50), 'learning_rate_init': 0.001}
  Fold 2
    SVM | Acc: 0.9781 | F1: 0.7170 | Params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
    KNN | Acc: 0.9833 | F1: 0.7139 | Params: {'n_neighbors': 3, 'weights': 'distance'}
    DecisionTree | Acc: 1.0000 | F1: 1.0000 | Params: {'max_depth': None, 'min_samples_split': 2}
    RandomForest | Acc: 1.0000 | F1: 1.0000 | Params: {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 100}
    MLP | Acc: 0.9948

/Users/Isaac/Documents/ML/MRMR/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


    MLP | Acc: 0.9843 | F1: 0.9809 | Params: {'alpha': 0.0001, 'hidden_layer_sizes': (100, 50), 'learning_rate_init': 0.001}
  Fold 10
    SVM | Acc: 0.9770 | F1: 0.9748 | Params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
    KNN | Acc: 0.9916 | F1: 0.7362 | Params: {'n_neighbors': 3, 'weights': 'distance'}
    DecisionTree | Acc: 1.0000 | F1: 1.0000 | Params: {'max_depth': None, 'min_samples_split': 2}
    RandomForest | Acc: 1.0000 | F1: 1.0000 | Params: {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 100}
    MLP | Acc: 0.9958 | F1: 0.9957 | Params: {'alpha': 0.0001, 'hidden_layer_sizes': (100, 50), 'learning_rate_init': 0.001}

Processing Dataset 10...
Path: ../data/10/train.csv
  Fold 1
    SVM | Acc: 0.9280 | F1: 0.4861 | Params: {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}
    KNN | Acc: 0.9328 | F1: 0.4960 | Params: {'n_neighbors': 5, 'weights': 'distance'}
    DecisionTree | Acc: 0.9376 | F1: 0.4935 | Params: {'max_depth': None, 'min_samples_split': 2}
    RandomFo

/Users/Isaac/Documents/ML/MRMR/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


    MLP | Acc: 0.9050 | F1: 0.5577 | Params: {'alpha': 0.0001, 'hidden_layer_sizes': (100, 50), 'learning_rate_init': 0.001}
  Fold 9
    SVM | Acc: 0.8883 | F1: 0.5595 | Params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
    KNN | Acc: 0.8956 | F1: 0.5964 | Params: {'n_neighbors': 3, 'weights': 'distance'}
    DecisionTree | Acc: 0.8914 | F1: 0.6033 | Params: {'max_depth': 20, 'min_samples_split': 5}
    RandomForest | Acc: 0.8935 | F1: 0.5939 | Params: {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 200}
    MLP | Acc: 0.8977 | F1: 0.5531 | Params: {'alpha': 0.0001, 'hidden_layer_sizes': (100, 50), 'learning_rate_init': 0.001}
  Fold 10
    SVM | Acc: 0.8935 | F1: 0.5541 | Params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
    KNN | Acc: 0.8987 | F1: 0.6205 | Params: {'n_neighbors': 3, 'weights': 'distance'}
    DecisionTree | Acc: 0.8956 | F1: 0.5995 | Params: {'max_depth': None, 'min_samples_split': 10}
    RandomForest | Acc: 0.9008 | F1: 0.6056 | Params: {'max_depth

...

    MLP | Acc: 0.9969 | F1: 0.9959 | Params: {'alpha': 0.001, 'hidden_layer_sizes': (100, 50), 'learning_rate_init': 0.001}
  Fold 3
    SVM | Acc: 0.9698 | F1: 0.7173 | Params: {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}
    KNN | Acc: 0.9823 | F1: 0.7289 | Params: {'n_neighbors': 5, 'weights': 'distance'}
    DecisionTree | Acc: 1.0000 | F1: 1.0000 | Params: {'max_depth': None, 'min_samples_split': 2}
    RandomForest | Acc: 1.0000 | F1: 1.0000 | Params: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}


/Users/Isaac/Documents/ML/MRMR/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/Isaac/Documents/ML/MRMR/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


    MLP | Acc: 0.9937 | F1: 0.9935 | Params: {'alpha': 0.0001, 'hidden_layer_sizes': (100, 50), 'learning_rate_init': 0.001}
  Fold 4
    SVM | Acc: 0.9687 | F1: 0.9519 | Params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
    KNN | Acc: 0.9916 | F1: 0.9882 | Params: {'n_neighbors': 5, 'weights': 'distance'}
    DecisionTree | Acc: 1.0000 | F1: 1.0000 | Params: {'max_depth': None, 'min_samples_split': 2}
    RandomForest | Acc: 1.0000 | F1: 1.0000 | Params: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
    MLP | Acc: 0.9958 | F1: 0.9942 | Params: {'alpha': 0.0001, 'hidden_layer_sizes': (100, 50), 'learning_rate_init': 0.001}
  Fold 5
    SVM | Acc: 0.9770 | F1: 0.9699 | Params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
    KNN | Acc: 0.9864 | F1: 0.7317 | Params: {'n_neighbors': 3, 'weights': 'distance'}
    DecisionTree | Acc: 1.0000 | F1: 1.0000 | Params: {'max_depth': None, 'min_samples_split': 2}
    RandomForest | Acc: 1.0000 | F1: 1.0000 | Params: {'max_dep

/Users/Isaac/Documents/ML/MRMR/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


    MLP | Acc: 0.9916 | F1: 0.9837 | Params: {'alpha': 0.0001, 'hidden_layer_sizes': (100, 50), 'learning_rate_init': 0.001}

Processing Dataset 8...
Path: ../data/8/train.csv
  Fold 1
    SVM | Acc: 0.9739 | F1: 0.9645 | Params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
    KNN | Acc: 0.9875 | F1: 0.9678 | Params: {'n_neighbors': 3, 'weights': 'distance'}
    DecisionTree | Acc: 1.0000 | F1: 1.0000 | Params: {'max_depth': None, 'min_samples_split': 2}
    RandomForest | Acc: 1.0000 | F1: 1.0000 | Params: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
    MLP | Acc: 0.9854 | F1: 0.9807 | Params: {'alpha': 0.0001, 'hidden_layer_sizes': (50,), 'learning_rate_init': 0.001}
  Fold 2
    SVM | Acc: 0.9791 | F1: 0.9676 | Params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
    KNN | Acc: 0.9927 | F1: 0.9905 | Params: {'n_neighbors': 3, 'weights': 'distance'}
    DecisionTree | Acc: 1.0000 | F1: 1.0000 | Params: {'max_depth': None, 'min_samples_split': 2}
    RandomForest

In [5]:
# CREATE RESULTS TABLE
results_df = pd.DataFrame(final_results)

# Sort nicely
results_df = results_df.sort_values(by=["Dataset", "Model"])

print("\n FINAL RESULTSn")
print(results_df.to_string(index=False))


# SAVE RESULTS
results_df.to_csv("final_phase1_results.csv", index=False)

print("\nResults saved to final_results.csv")


 FINAL RESULTSn
   Dataset        Model        Accuracy        F1 Score                                                                     Best Params
 Dataset_1 DecisionTree 1.0000 ± 0.0000 1.0000 ± 0.0000                                     {'max_depth': None, 'min_samples_split': 2}
 Dataset_1          KNN 0.9854 ± 0.0044 0.8487 ± 0.1244                                       {'n_neighbors': 3, 'weights': 'distance'}
 Dataset_1          MLP 0.9905 ± 0.0048 0.9613 ± 0.0761 {'alpha': 0.0001, 'hidden_layer_sizes': (100, 50), 'learning_rate_init': 0.001}
 Dataset_1 RandomForest 0.9996 ± 0.0007 0.9995 ± 0.0008                {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
 Dataset_1          SVM 0.9728 ± 0.0061 0.8783 ± 0.1111                                    {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Dataset_10 DecisionTree 0.9405 ± 0.0060 0.4780 ± 0.0340                                     {'max_depth': None, 'min_samples_split': 2}
Dataset_10          KNN 0.9346 